# Sprint 1 EDA — BM25 Failure Analysis

Goal: Identify which query classes BM25 struggles on and examine 10 concrete failure examples.

In [1]:
import json
import pandas as pd
from wands.retrieval.bm25 import BM25Retriever
from wands.eval.metrics import RELEVANCE

## 1. BM25 results by query class — worst performers

In [2]:
with open('../results/sprint1_bm25.json') as f:
    results = json.load(f)

rows = []
for qc, metrics in results['by_query_class'].items():
    rows.append({'query_class': qc, **metrics})

class_df = pd.DataFrame(rows).sort_values('ndcg@10')
print('=== Worst query classes by NDCG@10 ===')
class_df.head(10)

=== Worst query classes by NDCG@10 ===


,query_class,ndcg@10,mrr,recall@100,count
44,Vanities,0.000000,0.000000,0.000000,1
43,Unknown,0.000000,0.000000,0.000000,1
7,Bookcases,0.000000,0.023256,0.040984,1
8,Cabinet and Drawer Pulls,0.000000,0.076923,0.333333,1
9,Candle Holders,0.000000,0.071429,0.071429,1
15,Desks,0.000000,0.018182,0.004926,1
29,Patio Bar Stools,0.108630,0.500000,0.091837,1
28,Outdoor Conversation Sets,0.158197,0.357143,0.013549,3
40,Sofas,0.193426,0.125000,0.500000,1
49,Wall Décor,0.285209,1.000000,0.426966,1


In [3]:
print('=== Best query classes by NDCG@10 ===')
class_df.tail(10)

=== Best query classes by NDCG@10 ===


,query_class,ndcg@10,mrr,recall@100,count
21,Furniture Cushions,1.0,1.0,0.214511,1
45,Vanity Lighting,1.0,1.0,0.071429,1
1,Accent Pillows,1.0,1.0,0.096828,1
12,Curtains & Drapes,1.0,1.0,0.172117,1
39,Shower Heads,1.0,1.0,0.816327,1
30,Pendant Lights,1.0,1.0,0.240964,1
36,Sheets And Sheet Sets,1.0,1.0,0.862069,1
31,Plant & Telephone Tables,1.0,1.0,0.433180,1
41,String Lights,1.0,1.0,0.655172,1
50,Wall Plates,1.0,1.0,0.572254,1


## 2. Load data and retriever for failure inspection

In [4]:
retriever = BM25Retriever()

queries_df = pd.read_parquet('../data/processed/queries.parquet')
labels_df = pd.read_parquet('../data/processed/labels.parquet')
products_df = pd.read_parquet('../data/processed/products.parquet')

with open('../data/processed/splits.json') as f:
    splits = json.load(f)

val_queries = queries_df[queries_df['query_id'].isin(splits['val'])]
val_labels = labels_df[labels_df['query_id'].isin(splits['val'])]
print(f'{len(val_queries)} val queries')

72 val queries


## 3. Ten concrete failure examples

For each query where BM25 scored NDCG@10 = 0, show:
- The query text and class
- What BM25 returned (top 5)
- What the correct Exact matches were

In [5]:
from wands.eval.metrics import ndcg_at_k

failures = []
for _, row in val_queries.iterrows():
    qid = row['query_id']
    query_text = row['query']
    query_class = row['query_class']

    # Build relevance
    q_labels = val_labels[val_labels['query_id'] == qid]
    relevance = {r['product_id']: RELEVANCE[r['label']] for _, r in q_labels.iterrows()}

    # Get BM25 results
    predicted = retriever.search(query_text, k=100)
    score = ndcg_at_k(predicted, relevance, k=10)

    if score < 0.3:
        # Get the Exact matches BM25 should have found
        exact_ids = [pid for pid, rel in relevance.items() if rel == 2]
        exact_products = products_df[products_df['product_id'].isin(exact_ids)]['product_name'].tolist()

        # Get what BM25 actually returned
        top5_ids = predicted[:5]
        top5_products = products_df[products_df['product_id'].isin(top5_ids)]['product_name'].tolist()

        failures.append({
            'query': query_text,
            'query_class': query_class,
            'ndcg@10': round(score, 4),
            'bm25_top5': top5_products,
            'expected_exact': exact_products[:5],
        })

print(f'Found {len(failures)} low-scoring queries (NDCG@10 < 0.3)\n')
for i, f in enumerate(failures[:10]):
    print(f"--- Failure {i+1} ---")
    print(f"Query: '{f['query']}' (class: {f['query_class']})")
    print(f"NDCG@10: {f['ndcg@10']}")
    print(f"BM25 returned: {f['bm25_top5']}")
    print(f"Expected (Exact): {f['expected_exact']}")
    print()

Found 19 low-scoring queries (NDCG@10 < 0.3)

--- Failure 1 ---
Query: 'front porch furniture sets' (class: Outdoor Conversation Sets)
NDCG@10: 0.0
BM25 returned: ['front porch welcome sign wall décor', 'ge appliances smart 4.8 cu . ft. front load washer and 7.8 cu . ft. electric dryer', 'ge appliances smart 4.8 cu . ft. front load washer and 7.8 cu . ft. gas dryer', '4.5 cu . ft. front load washer and 7.5 cu . ft. electric dryer', '4.5 cu . ft. smart front load washer and 7.5 cu . ft. electric smart dryer']
Expected (Exact): ['3 piece rattan sofa seating group with cushions', 'ewout 7 piece sectional seating group with cushions', 'ewura 6 piece sectional seating group with cushions', 'augusta 6 piece rattan sectional seating group with cushions', 'outdoor 4 piece aluminum sofa conversation set , patio furniture seating group with cushions']

--- Failure 2 ---
Query: 'black hanging bookshelf' (class: Bookcases)
NDCG@10: 0.0
BM25 returned: ['bathroom double towel bar towel hanging rod/r

## 4. Summary

Key observations (fill in after running):

1. **Which query classes fail?** — 
2. **Why does BM25 fail?** — Common causes: vocabulary mismatch (query uses different terms than product), vague/style queries, category confusion
3. **What would help?** — Dense retrieval (semantic matching), query parsing (structured constraints), reranking